# 04. Feature Engineering(実行)

ベースライン(約 0.9412)から **+0.003〜0.004** を積んだ工程。効いたものと効かなかったものを
実際に動かして確認する。関数の中身と**各モデルの最終的な列一覧**は `03_fe.ipynb` を参照。

## 結論を先に

「本番」は 5 fold・収束まで学習・Triple TE(smooth 3通り)× キー 17 本。
「本 NB」は 2 fold・単一 smooth・キー 13 本の縮小版なので、値は本番より小さく出る。

| 施策 | 本番での効果 | 本 NB の実測 | 理由 |
|---|---|---|---|
| **厳密値 Target Encoding** | **+0.003 前後** | **+0.00177** | 最大の改善要因 |
| Count Encoding | +0.0005〜0.0008 | +0.00029 | TE と非冗長で加算的(CatBoost では無効) |
| Triple TE + digit + ビン1024 | +0.0005〜0.001 | (本 NB では未実施) | 上位カーネル由来 |
| 四則演算 (diff/ratio/sum/avg) | **無効〜悪化** | **-0.00003** | 生成過程に交互作用がない |
| 交互作用 TE(2〜13列) | **すべて無効** | (本 NB では未実施) | 同上 |


In [1]:
import os, sys
# リポジトリルートを作業ディレクトリにして、data/ などの相対パスを揃える
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

cwd: C:\Users\takac\dev\python\kaggle\2609


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

TARGET = "Will_Buy_EV"
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
y = (train[TARGET] == "Yes").astype(int)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

NUM_COLS = [c for c in train.select_dtypes(include=[np.number]).columns if c != "id"]
CAT_COLS = [c for c in train.columns if c not in NUM_COLS + ["id", TARGET]]
ALL_COLS = NUM_COLS + CAT_COLS

## なぜ Target Encoding が効くのか

EDA で見たとおり年収は 13,214 種類の値を持つ。木は既定で 255 ビンにまとめるため、
**値ごとに違う購入率を直接は学べない**。TE はその「値ごとの購入率」を 1 列で渡す。

数値列もビン分割せず **厳密な値のままキーにする**のが要点。

In [3]:
rate = y.groupby(train["Annual_Income_USD"]).agg(["mean", "size"])
print("年収のユニーク値:", len(rate))
print("1値あたりの平均行数:", round(rate["size"].mean(), 1))
rate.head()

年収のユニーク値: 13214
1値あたりの平均行数: 50.6


,mean,size
Annual_Income_USD,,
30000.0,0.044266,61605
31003.0,0.500000,2
38174.0,0.000000,1
38209.0,0.000000,1
38250.0,0.000000,1


## リーク対策 — 入れ子 CV

TE は目的変数を使うため、作り方を誤ると学習データの答えが漏れる。ここでは二重に防ぐ。

1. 外側 fold の学習データ内だけで統計を計算する
2. **学習行には内側 CV の out-of-fold 値**を当てる(自分のラベルを含まない値にする)

2 は単なる保険ではなく、XGBoost では **+0.00108** の精度向上につながった。
学習時だけ TE が当たりすぎる状態(楽観バイアス)が解消されるため。

In [4]:
def target_encode(tr_key, tr_y, va_key, prior, smooth=20.0, inner_splits=5):
    """外側foldの学習データで fit し、学習行には内側OOF値を当てる入れ子TE。"""
    # valid/test 用: 学習fold全体の統計
    agg = tr_y.groupby(tr_key).agg(["sum", "count"])
    m = (agg["sum"] + prior * smooth) / (agg["count"] + smooth)
    va_te = va_key.map(m).fillna(prior).astype("float32")

    # 学習行用: 内側CVのOOF値
    tr_te = pd.Series(np.full(len(tr_key), prior, dtype="float32"), index=tr_key.index)
    inner = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=42)
    for i_tr, i_va in inner.split(tr_key, tr_y):
        a = tr_y.iloc[i_tr].groupby(tr_key.iloc[i_tr]).agg(["sum", "count"])
        mm = (a["sum"] + prior * smooth) / (a["count"] + smooth)
        tr_te.iloc[i_va] = tr_key.iloc[i_va].map(mm).fillna(prior).astype("float32").values
    return tr_te, va_te

## 評価用の CV ループ

特徴量の作り方(`build`)を差し替えて A/B するための関数。
`build` は fold ごとに呼ばれ、学習用と検証用の特徴量を返す。

**モデル設定は本番に寄せる**(深さ5・列サンプリング 0.3・ビン数 1024・lr 0.05)。
ここは手を抜けない。素の LightGBM(深さ無制限・列サンプリングなし・lr 0.1・300本)で
同じ A/B をやると、**厳密値 TE が -0.0083 と悪化して結論が逆に出る**。
年収のような高カーディナリティ列の TE は 1 値あたりの行数が少なく分散が大きいので、
正則化が効いていない木はそのノイズに飛びついてしまう。
TE が効くのは「木を弱くしてから」という順序に意味がある。


In [5]:
# 本番の設定(深さ5 / 列0.3 / ビン1024)。ここを緩めると TE の効果が逆転する。
DEMO_PARAMS = dict(n_estimators=1500, learning_rate=0.05, max_depth=5,
                   colsample_bytree=0.3, max_bin=1024)


def evaluate(build, n_splits=5, seed=42, **over):
    params = dict(DEMO_PARAMS, **over)
    oof = np.zeros(len(train))
    mask = np.zeros(len(train), dtype=bool)
    for fold, (tr, va) in enumerate(skf.split(train, y)):
        Xtr, Xva = build(tr, va)
        evaluate.last_cols = list(Xtr.columns)   # 実際に学習へ渡した列を控えておく
        model = LGBMClassifier(random_state=seed, verbosity=-1, **params)
        model.fit(Xtr, y.iloc[tr])
        oof[va] = model.predict_proba(Xva)[:, 1]
        mask[va] = True
        if fold + 1 >= n_splits:
            break
    score = roc_auc_score(y[mask], oof[mask])
    print(f"AUC: {score:.5f}  (列数 {len(evaluate.last_cols)})")
    return score


evaluate.last_cols = []
COLS_LOG = {}          # ステップ名 -> 使った列(あとで一覧表にする)


def log_cols(label, prev=None, show=8):
    """直前の evaluate() が使った列を記録し、前ステップからの増分を表示する。"""
    cols = list(evaluate.last_cols)
    COLS_LOG[label] = cols
    added = [c for c in cols if prev is None or c not in COLS_LOG.get(prev, [])]
    print(f"{chr(10)}[{label}] 合計 {len(cols)} 列 / このステップで追加 {len(added)} 列")
    if added:
        for i in range(0, min(len(added), show), 4):
            print("   ", "  ".join(f"{c:<30}" for c in added[i:i + 4]).rstrip())
        if len(added) > show:
            print(f"    … 他 {len(added) - show} 列")
    return cols


## A. ベースライン(生の特徴量のみ)

時間短縮のため **2 fold** で比較する。数値は本番(5 fold・収束まで学習)とは揃わないが、
施策同士の比較には使える。1 ステップおよそ 2〜4 分。


In [6]:
def prep_cat(df):
    out = df.copy()
    for c in CAT_COLS:
        cats = pd.concat([train[c], test[c]]).astype("category").cat.categories
        out[c] = pd.Categorical(out[c], categories=cats)
    return out

X_raw = prep_cat(train[ALL_COLS])

def build_base(tr, va):
    return X_raw.iloc[tr], X_raw.iloc[va]

score_base = evaluate(build_base, n_splits=2)
log_cols("A. 生の特徴量");


AUC: 0.94273  (列数 13)

[A. 生の特徴量] 合計 13 列 / このステップで追加 13 列
    Age                             Annual_Income_USD               Daily_Commute_km                Number_of_Cars_Owned
    Charging_Stations_Near_Home     Charging_Stations_Near_Work     Environmental_Concern_Level     Gender
    … 他 5 列


## B. + 厳密値 Target Encoding(全13列)

In [7]:
prior = y.mean()

def build_te(tr, va):
    Xtr, Xva = X_raw.iloc[tr].copy(), X_raw.iloc[va].copy()
    for c in ALL_COLS:
        key = train[c].astype(str)
        t, v = target_encode(key.iloc[tr], y.iloc[tr], key.iloc[va], prior)
        Xtr[f"te_{c}"], Xva[f"te_{c}"] = t.values, v.values
    return Xtr, Xva

score_te = evaluate(build_te, n_splits=2)
print(f"ベースラインとの差: {score_te - score_base:+.5f}")
log_cols("B. + 厳密値TE", prev="A. 生の特徴量");


AUC: 0.94450  (列数 26)
ベースラインとの差: +0.00177

[B. + 厳密値TE] 合計 26 列 / このステップで追加 13 列
    te_Age                          te_Annual_Income_USD            te_Daily_Commute_km             te_Number_of_Cars_Owned
    te_Charging_Stations_Near_Home  te_Charging_Stations_Near_Work  te_Environmental_Concern_Level  te_Gender
    … 他 5 列


## C. + Count Encoding

値の出現頻度。**目的変数を使わないので train+test をまとめて数えてよい**(リークしない)。
TE とは別の情報なので加算的に効く。

In [8]:
count_maps = {c: pd.concat([train[c], test[c]]).astype(str).value_counts() for c in ALL_COLS}

def build_te_cnt(tr, va):
    Xtr, Xva = build_te(tr, va)
    for c in ALL_COLS:
        key = train[c].astype(str)
        Xtr[f"cnt_{c}"] = key.iloc[tr].map(count_maps[c]).values
        Xva[f"cnt_{c}"] = key.iloc[va].map(count_maps[c]).values
    return Xtr, Xva

score_te_cnt = evaluate(build_te_cnt, n_splits=2)
print(f"TE のみとの差: {score_te_cnt - score_te:+.5f}")
log_cols("C. + Count Encoding", prev="B. + 厳密値TE");


AUC: 0.94479  (列数 39)
TE のみとの差: +0.00029

[C. + Count Encoding] 合計 39 列 / このステップで追加 13 列
    cnt_Age                         cnt_Annual_Income_USD           cnt_Daily_Commute_km            cnt_Number_of_Cars_Owned
    cnt_Charging_Stations_Near_Home  cnt_Charging_Stations_Near_Work  cnt_Environmental_Concern_Level  cnt_Gender
    … 他 5 列


## D. + 四則演算(効かない例)

「充電スタンド数の自宅+職場」「年収÷通勤距離」など直感的な合成指標。
木は 1 列ずつしか分割できないので効きそうに見えるが、**合成データの生成過程に
そうした関係がない**ため効かない。3モデルすべてで無効〜悪化だった。

In [9]:
PAIRS = [("Charging_Stations_Near_Home", "Charging_Stations_Near_Work"),
         ("Annual_Income_USD", "Daily_Commute_km"),
         ("Age", "Annual_Income_USD")]

def build_arith(tr, va):
    Xtr, Xva = build_te_cnt(tr, va)
    for a, b in PAIRS:
        for X_, idx in ((Xtr, tr), (Xva, va)):
            X_[f"{a}_sum_{b}"] = train[a].iloc[idx].values + train[b].iloc[idx].values
            X_[f"{a}_diff_{b}"] = train[a].iloc[idx].values - train[b].iloc[idx].values
            X_[f"{a}_ratio_{b}"] = train[a].iloc[idx].values / (train[b].iloc[idx].values + 1e-6)
    return Xtr, Xva

score_arith = evaluate(build_arith, n_splits=2)
print(f"TE+Count との差: {score_arith - score_te_cnt:+.5f}")
log_cols("D. + 四則演算", prev="C. + Count Encoding");


AUC: 0.94476  (列数 48)
TE+Count との差: -0.00003

[D. + 四則演算] 合計 48 列 / このステップで追加 9 列
    Charging_Stations_Near_Home_sum_Charging_Stations_Near_Work  Charging_Stations_Near_Home_diff_Charging_Stations_Near_Work  Charging_Stations_Near_Home_ratio_Charging_Stations_Near_Work  Annual_Income_USD_sum_Daily_Commute_km
    Annual_Income_USD_diff_Daily_Commute_km  Annual_Income_USD_ratio_Daily_Commute_km  Age_sum_Annual_Income_USD       Age_diff_Annual_Income_USD
    … 他 1 列


## 結果のまとめ

In [10]:
LABELS = ["A. 生の特徴量", "B. + 厳密値TE", "C. + Count Encoding", "D. + 四則演算"]
pd.DataFrame({
    "AUC": [score_base, score_te, score_te_cnt, score_arith],
    "前からの差": [np.nan, score_te - score_base, score_te_cnt - score_te, score_arith - score_te_cnt],
    "列数": [len(COLS_LOG[l]) for l in LABELS],
}, index=LABELS).round(5)


,AUC,前からの差,列数
A. 生の特徴量,0.94273,NaN,13
B. + 厳密値TE,0.94450,0.00177,26
C. + Count Encoding,0.94479,0.00029,39
D. + 四則演算,0.94476,-0.00003,48


### このノートブックで作った列の全一覧

上の A〜D で何が増えたのかを、列名まで展開して確認する。
**四則演算(D)は 9 列も増えるのに AUC が動かない**ことが目で見て分かる。


In [11]:
for label in LABELS:
    cols = COLS_LOG[label]
    prev = LABELS[LABELS.index(label) - 1] if LABELS.index(label) else None
    added = [c for c in cols if prev is None or c not in COLS_LOG[prev]]
    print(f"\n■ {label}  —  合計 {len(cols)} 列 / 追加 {len(added)} 列")
    for i in range(0, len(added), 3):
        print("    " + "  ".join(f"{c:<38}" for c in added[i:i + 3]).rstrip())




■ A. 生の特徴量  —  合計 13 列 / 追加 13 列
    Age                                     Annual_Income_USD                       Daily_Commute_km
    Number_of_Cars_Owned                    Charging_Stations_Near_Home             Charging_Stations_Near_Work
    Environmental_Concern_Level             Gender                                  City_Type
    Current_Car_Type                        Home_Charging_Possible                  Subsidy_Available
    Range_Anxiety_Level

■ B. + 厳密値TE  —  合計 26 列 / 追加 13 列
    te_Age                                  te_Annual_Income_USD                    te_Daily_Commute_km
    te_Number_of_Cars_Owned                 te_Charging_Stations_Near_Home          te_Charging_Stations_Near_Work
    te_Environmental_Concern_Level          te_Gender                               te_City_Type
    te_Current_Car_Type                     te_Home_Charging_Possible               te_Subsidy_Available
    te_Range_Anxiety_Level

■ C. + Count Encoding  —  合計 39 列 / 追加 13 列
    

## 本番の構成

ここまでの結果に、上位カーネル由来の施策を足したものが本番。

- **Triple TE**: smooth(縮約の強さ)を変えた TE を 3 本**同時に**入れる
- **Smooth Keys**: 年収を /10、/100、/1000 に丸めたものも TE のキーにする
- **digit features**: 数値を桁ごとにばらして列にする
- **ビン数 1024**: 既定の 255 では年収の細かい違いが潰れるため

本番の実行コマンドは README の「現行ベストの再現コマンド」を参照
(`src/04_fe_run_<model>.py`)。**各モデルが最終的に使っている列の一覧は
`03_fe.ipynb` の「各モデルが実際に使っている列(本番構成)」にある。**

| モデル | 列数 | OOF AUC | ベースライン | 差 |
|---|---|---|---|---|
| LightGBM | 92 | 0.94610 | 0.94123 | +0.00487 |
| XGBoost | 93 | 0.94608 | 0.94124 | +0.00484 |
| CatBoost | 80 | 0.94589 | 0.94156 | +0.00433 |
| RealMLP | 38 | 0.94589 | — | — |
| **3モデルアンサンブル(LGBM+XGB+RealMLP)** | — | **0.94623** | — | — |

CatBoost は単体では 3 番手タイだが、他の GBDT と予測が同質(rank 相関 0.999)なため
貪欲法のブレンドで重み 0 になった。逆に列構成がまったく違う RealMLP は単体が同じ 0.94589 でも
採用されている。**列の中身が違うことがアンサンブルでの価値になる。**
